# LoD1 Timing and Quality Considerations

[![Binder](_static/launch-binder.svg)](https://mybinder.org/v2/gh/AdrianKriger/geo3D_wrkshp/HEAD?urlpath=%2Fdoc%2Ftree%2Fworkshop%2Fnotebooks%2CityJSONLoD1timing.ipynb)

<div class="alert alert-block alert-warning"><b>This notebook will:</b>

> **illustrate how different resolution elevation models (25m, 15m and 10m DEM) affect;**
>>
>> **1) `osm_LoD1_3DCityModel.ipynb` timing and**<br> 
>> **2) the quality** *(holes and completeness)* **of a LoD1 3D City Model**
</div>

In [1]:
#- load the magic
import time
from datetime import timedelta
import tempfile

import os
from itertools import chain
import math
import requests
import overpass
#import osm2geojson
import copy
import json
import numpy as np
import pandas as pd
#import geopandas as gpd
import topojson as tp

import shapely
from shapely.geometry import Point, Polygon, polygon 
from shapely.ops import snap, transform
from shapely.strtree import STRtree

import city3D

import pyproj
from osgeo import gdal, ogr, osr

import triangle as tr

from openlocationcode import openlocationcode as olc

import matplotlib.pyplot as plt

In [2]:
Tstart = time.time()
import warnings
warnings.filterwarnings('ignore')

**A `parameter.json` defines the path and files**.

In [3]:
#jparams = json.load(open('sRiver_param10m.json'))  # was 11 min 30 sec | now 0:05:12.30
jparams = json.load(open('sRiver_param15m.json'))  # was 5 min 20 sec | now 0:02:24.49
#jparams = json.load(open('sRiver_param25m.json'))  # was 2 min 13 sec | now 0:01:00.15

| area of interest | elevation model | CityJSON and metadata |
|:--------:|:--------:|:--------:|
|![area.png](_static/area.png)|![raster.png](_static/raster15.png)|![meta.png](_static/meta15.png) |

In [4]:
#- input OSM PBF file
input_pbf = "./data/CapeTown.osm.pbf"

**Lets first harvest the boundary of the area; we want to interogate**

In [5]:
#- get the area [suburb]
query = """[out:json][timeout:30];
        area[boundary=administrative][name='{0}'] -> .a;
        (
        way[amenity~'university|research_institute'][name='{1}'](area.a);
        relation[place][place~"sub|town|city|count|state|village|borough|quarter|neighbourhood"][name='{1}'](area.a);
        );
        out geom;
        """.format(jparams['LargeArea'], jparams['FocusArea'])

#url = "http://overpass-api.de/api/interpreter"
#r = requests.get(url, params={'data': query})
#rr = r.read()
#area = osm2geojson.json2geojson(r.json())
#read into .gpd
#aoi = gpd.GeoDataFrame.from_features(area['features'])
#aoi = aoi.set_crs(4326, allow_override=True)
#if jparams['osm_type'] == 'relation' and len(aoi) > 1:
#    for i, row in aoi.iterrows():
#        if row.tags != None and 'place' in row.tags:
#            focus = row
            
#    trim = pd.DataFrame(focus)
#    trim = trim.T
#    aoi = gpd.GeoDataFrame(trim, geometry = trim['geometry'])
    #aoi = aoi.set_crs(4326)

# Drop rows where geometry is None or NaN
#aoi = aoi.dropna(subset=['geometry'])
#aoi = aoi.set_crs(4326, allow_override=True)

#- execute function from city3D and and return GeoDataFrameLite | home-baked gdf
aoi = city3D.overpass_to_gdf(query)

#- suppose 'aoi' is your GeoDataFrameLite or list of geometries
geoms = aoi['geometry'].tolist()
#- combine all geometries into a single union
combined_geom = shapely.unary_union(geoms)  # returns Polygon or MultiPolygon
#- compute bounding box
minx, miny, maxx, maxy = combined_geom.bounds
#extent = [minx - 250, miny - 250,maxx + 250, maxy + 250]
aoi.head(2)

,boundary,name,place,type,wikidata,geometry,osm_id,osm_type
0,place,Salt River,suburb,boundary,Q2383969,"POLYGON ((18.4575251 -33.9411991, 18.4591488 -...",2034284,relation


**Only harvest what we need from the osm.pbf.**

In [6]:
start = time.time()

gdal.UseExceptions()
gdal.SetConfigOption("OGR_GEOMETRY_ACCEPT_UNCLOSED_RING", "NO") 
#gdal.SetConfigOption("USE_CUSTOM_INDEXING", "NO")
# Input OSM PBF file
#input_pbf = "your_data.osm.pbf"

# GDAL Virtual File System (VSI) to avoid writing to disk
geojson_vsimem = "/vsimem/temp.geojson"

#- GDAL VectorTranslate to extract only buildings & fix geometries
gdal.VectorTranslate(
    geojson_vsimem,                                           # Output as in-memory GeoJSON
    input_pbf,                                                # Source OSM PBF file
    format="GeoJSON",                                         # Output format
    layers=["multipolygons"],                                 # Extract only multipolygons
    options=["-where", "building IS NOT NULL", "-makevalid", 
                          "-spat", str(minx), str(miny), str(maxx), str(maxy)]  # Filter buildings & fix geometries
)

#- load into GeoDataFrame
#- load into GeoDataFrameLite | home-baked gdf
#gdf = gpd.read_file(geojson_vsimem)
gdf = city3D.read_vsimem_geojson(geojson_vsimem)

#- cleanup VSI Memory
gdal.Unlink(geojson_vsimem)

# show gdf
#gdf.head()

end = time.time()
print('runtime:', str(timedelta(seconds=(end - start))))

runtime: 0:00:01.464370


In [7]:
gdf.head(2)
#len(gdf)

,amenity,building,craft,geometry,historic,leisure,man_made,name,office,osm_id,osm_way_id,other_tags,shop,sport,tourism,type
0,None,office,None,"MULTIPOLYGON (((18.4723433 -33.9288948, 18.472...",None,None,None,Western Cape Metrorail - Infrastructure Building,None,6383946,None,"""addr:city""=>""Cape Town"",""addr:postcode""=>""729...",None,None,None,multipolygon
1,None,hall,None,"MULTIPOLYGON (((18.4686004 -33.9360187, 18.468...",None,None,None,None,None,6691666,None,"""addr:housename""=>""Parish Hall""",None,None,None,multipolygon


In [8]:
# Convert valid strings, ignore None/NaN
def safe_convert(tag_string):
    if isinstance(tag_string, str):
        try:
            # Replace "=>" with ":" and fix newlines
            formatted_string = "{" + tag_string.replace("=>", ":").replace("\n", " ") + "}"
            return json.loads(formatted_string)  # Parse safely
        except json.JSONDecodeError:
            return {}  # Return empty dict on failure
    return {}  # Return empty dict if NaN or None

# Apply conversion function
gdf["tags"] = gdf["other_tags"].apply(safe_convert)

# Normalize the 'tags' column to create a new DataFrame
tags_df = pd.json_normalize(gdf['tags'])
# Join the new columns back to the original GeoDataFrame
gdf = pd.concat([gdf, tags_df], axis=1)
# (Optional) Drop the original 'tags' column
gdf = gdf.drop(columns=['other_tags'])

# Ensure a single 'osm_id' column
if 'osm_id' in gdf.columns:
    if 'osm_way_id' in gdf.columns:
        gdf['osm_id'] = [o if pd.notna(o) else w 
                         for o, w in zip(gdf['osm_id'], gdf['osm_way_id'])]
        gdf = gdf.drop(columns=['osm_way_id'])
elif 'osm_way_id' in gdf.columns:
    gdf = gdf.rename(columns={'osm_way_id': 'osm_id'})

#gdf = gdf[gdf.geometry.apply(lambda x: x.within(aoi.unary_union))]
gdf = gdf[gdf.geometry.apply(lambda x: x.within(shapely.unary_union(aoi.geometry)))]
gdf.crs = "EPSG:4326"

gdf.head(2)

,amenity,building,craft,geometry,historic,leisure,man_made,name,office,osm_id,...,diet:vegan,content,industrial,addr:unit,disused,sahra:criterea,opening_date,bus,network,construction
0,None,office,None,"MULTIPOLYGON (((18.4723433 -33.9288948, 18.472...",None,None,None,Western Cape Metrorail - Infrastructure Building,None,6383946,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,None,school,None,"MULTIPOLYGON (((18.4615816 -33.9317448, 18.461...",None,None,None,None,None,13328172,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
ts = gdf[gdf['building'].notna()]
#len(ts)
print('\n', len(ts), "buildings have been harvested from", input_pbf)


 1449 buildings have been harvested from ./data/CapeTown.osm.pbf


In [10]:
#ts.head(2)

In [11]:
# basic cleaning to harvest building=* (no building:part=*) and building=levels tags only

#- we only want buildings with =levels data
ts = (
    ts.dropna(subset=['building:levels'])
      .assign(**{'building:levels': pd.to_numeric(ts['building:levels'], errors='coerce')})
      .query("`building:levels` != 0")
)

#- without building:part
#ts = ts[ts['building:part'].isnull()]
ts = ts[ts['building:part'].isnull()] if 'building:part' in ts.columns else ts
#ts = ts.explode()
print('\n\033[1m', jparams['FocusArea'], 'has \033[0m', len(ts), 'buildings')


 Salt River has  1377 buildings


In [12]:
#- fill <Projected CRS: EPSG:32734> from above here epsg = EPSG:32734
epsg = 'EPSG:32734'

In [13]:
#project blds
ts = ts.to_crs(epsg)
#project aoi
aoi = aoi.to_crs(epsg)

**Create LoD1 3D City Model**

In [14]:
aoibuffer = aoi.copy()

def buffer01(row):
    with np.errstate(invalid='ignore'):
        return row.geometry.buffer(150, cap_style=3, join_style=2)

aoibuffer['geometry'] = aoibuffer.apply(buffer01, axis=1)
    
#extent = [aoibuffer.total_bounds[0] - 250, aoibuffer.total_bounds[1] - 250, 
#          aoibuffer.total_bounds[2] + 250, aoibuffer.total_bounds[3] + 250]


#- suppose 'aoi' is your GeoDataFrameLite or list of geometries
geoms = aoibuffer['geometry'].tolist()
#- combine all geometries into a single union
combined_geom = shapely.unary_union(geoms)  # returns Polygon or MultiPolygon
#- compute bounding box
minx, miny, maxx, maxy = combined_geom.bounds

extent = [minx - 250, miny - 250,
          maxx + 250, maxy + 250]

**Now the DEM**  
*one is available at [raster](https://github.com/AdrianKriger/geo3D/suburb/tree/main/raster)*

In [15]:
gdal.SetConfigOption("GTIFF_SRS_SOURCE", "GEOKEYS")
gdal.UseExceptions() 

# set the path and nodata
OutTile = gdal.Warp(jparams['projClip_raster'], 
                    jparams['in_raster'],
                    dstSRS=epsg,
                    srcNodata = jparams['nodata'],
                    #-  dstNodata = 0,
                    #-- outputBounds=[minX, minY, maxX, maxY]
                    outputBounds = [extent[0], extent[1], extent[2], extent[3]])
OutTile = None 

In [16]:
#- convert raster to XYZ in-memory
#- Virtual in-memory path
xyz_mem_path = "/vsimem/temp_xyz.xyz"  
gdal.Translate(xyz_mem_path, jparams['projClip_raster'], format="XYZ")  

#0 read XYZ from GDAL's in-memory file
xyz_vsimem = gdal.VSIFOpenL(xyz_mem_path, "rb")
xyz_bytes = gdal.VSIFReadL(1, gdal.VSIStatL(xyz_mem_path).size, xyz_vsimem)
gdal.VSIFCloseL(xyz_vsimem)
#- cleanup in-memory file
gdal.Unlink(xyz_mem_path) 

0

**prepare to harvest elevation**

In [17]:
# set the path to the projected, cliped elevation
src_filename = jparams['projClip_raster']

src_ds = gdal.Open(src_filename) 
gt_forward = src_ds.GetGeoTransform()
rb = src_ds.GetRasterBand(1)

def rasterQuery(geom, gt_forward, rb):

    mx = geom.representative_point().x
    my = geom.representative_point().y
    
    px = int((mx - gt_forward[0]) / gt_forward[1])
    py = int((my - gt_forward[3]) / gt_forward[5])

    intval = rb.ReadAsArray(px, py, 1, 1)
 
    return intval[0][0]

**Buildings**

In [18]:
#- simplify geometry with GeoDataFrameLite | home-baked gdf
ts = city3D.GeoDataFrameLite(ts)
geojson_dict = json.loads(ts.to_json())

for feat in geojson_dict["features"]:
    if feat.get("type") is None:
        feat["type"] = "multipolygon"
    if feat.get("geometry") is None:
        feat["geometry"] = {"type":"MultiPolygon","coordinates":[]}

topo = tp.Topology(geojson_dict, prequantize=False, winding_order='CCW_CW')
simplified_geojson = topo.toposimplify(0.25).to_geojson()

ts = city3D.GeoDataFrameLite.from_json(simplified_geojson)
ts.crs = epsg

In [19]:
# prepare to plot (more buildings = more time) 
start = time.time()

ts_copy = ts.copy()
#new_df1 = ts_copy.loc[ts_copy.overlaps(ts_copy.unary_union)].reset_index(drop=True)  #-- perhaps no union?

#joined = gpd.sjoin(ts_copy, ts_copy, how="inner", predicate="overlaps")

geoms = ts_copy["geometry"].tolist()
tree = STRtree(geoms)
# Find overlapping pairs
pairs = []
for i, geom in enumerate(geoms):
    # query returns candidates that intersect envelope
    candidates = tree.query(geom)
    for c in candidates:
        # find index by identity in the original geoms list
        try:
            j = next(idx for idx, g in enumerate(geoms) if g is c)
        except StopIteration:
            continue  # skip if candidate is not found
        if i != j and geom.overlaps(c):
            pairs.append((i, j))

joined = pd.DataFrame(pairs, columns=["index_left", "index_right"])

#- joined index is changing. silence warning. future ready.
# GeoPandas <0.12 uses index_left/index_right
#if "index_left" in joined.columns:
#    left_idx = joined["index_left"]
#    right_idx = joined["index_right"]
#else:  # GeoPandas ≥0.12 uses _left/_right suffixes
#    left_idx = joined.index
#    right_idx = joined["index_right"]

# remove self matches
#mask = left_idx != right_idx
#overlap_idx = left_idx[mask].unique()

#- remove self matches and select overlapping geometries
left_idx = joined["index_left"]
right_idx = joined["index_right"]
mask = left_idx != right_idx
overlap_idx = left_idx[mask].unique()

new_df1 = ts_copy.loc[overlap_idx].reset_index(drop=True)

end = time.time()
print('runtime:', str(timedelta(seconds=(end - start))))

runtime: 0:00:00.160511


**Plot**

*Browse the saved `'./data/topologyFig'` at your leisure*

In [20]:
#%matplotlib

#fig, ax = plt.subplots(figsize=(11, 11))
#ts.plot(ax=ax, facecolor='none', edgecolor='purple', alpha=0.2)
#if len(new_df1) > 0:
#    new_df1.plot(ax=ax, edgecolor='red', facecolor='none')#, alpha=0.3)#, column='osm_building', legend=True)

#def plot_geometries(df, ax=None, facecolor='none', edgecolor='purple', alpha=0.5):
#    if ax is None:
#        fig, ax = plt.subplots(figsize=(10,10))

#    patches = []

#    for geom in df['geometry']:
#        if geom is None:
#            continue

#        if isinstance(geom, Polygon):
            # Exterior ring
#            patches.append(MplPolygon(list(geom.exterior.coords), closed=True))
            # Interiors (holes)
#            for interior in geom.interiors:
#                patches.append(MplPolygon(list(interior.coords), closed=True))
#        elif isinstance(geom, MultiPolygon):
#            for poly in geom.geoms:
#                patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
#                for interior in poly.interiors:
#                    patches.append(MplPolygon(list(interior.coords), closed=True))

#    pc = PatchCollection(patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha)
#    ax.add_collection(pc)
#    ax.autoscale()
#    ax.set_aspect('equal')
#    return ax

##-- Example usage:
#fig, ax = plt.subplots(figsize=(11, 11))
#plot_geometries(ts_copy, ax=ax, facecolor='none', edgecolor='purple', alpha=0.2)
#if len(new_df1) > 0:
#    plot_geometries(new_df1, ax=ax, facecolor='none', edgecolor='red', alpha=0.5)
#-- save
#plt.savefig('./data/topologyFig', dpi=300)
#plt.show()

In [21]:
#- get the mean height of the bld
ts['mean'] = ts.apply(lambda row: rasterQuery(row.geometry, gt_forward, rb), axis = 1)
ts.head(2)

,amenity,building,craft,historic,leisure,man_made,name,office,osm_id,shop,...,industrial,addr:unit,disused,sahra:criterea,opening_date,bus,network,construction,geometry,mean
0,None,office,None,None,None,None,Western Cape Metrorail - Infrastructure Building,None,6383946,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON (((266353.2773689979 6242850.2127...,4.306250
1,None,school,None,None,None,None,None,None,13328172,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON (((265366.0837814461 6242509.5266...,16.895311


In [22]:
#ts['building:levels'].unique

In [23]:
# -- execute function. write geoJSON
dis = city3D.process_and_write_geojson(ts, jparams)

In [24]:
start = time.time()

#dis = gpd.read_file(jparams['osm_bldings'])                   
#dis.set_crs(epsg=int(epsg[-5:]), inplace=True, allow_override=True)

#dict_vertices = {}
#cols = [c for c in ['bottom_bridge_height', 'bottom_roof_height', 'roof_height'] if c in dis.columns]

#dis['geometry'] = dis.geometry.apply(polygon.orient, 1)

#for i, row in dis.iterrows():
#    oring = list(row.geometry.exterior.coords)
#    name = row['osm_id']
#    for (j, v) in enumerate(oring[:-1]):
#        vertex = (oring[j][0], oring[j][1])
#        attr = [row[c] for c in cols]
#        attr = [x for x in attr if not np.isnan(x)]  # Remove np.nan values
#        if vertex in dict_vertices.keys():
#            dict_vertices[vertex][row['osm_id']] = attr
#        else:
#            dict_vertices[vertex] = {row['osm_id']: attr}

#result = {}
#for k1, d in dict_vertices.items():
#    for k2 in d:
#        result.setdefault(k2, {})[k1] = sorted(list(set([j for i in d.values() for j in i])))
        
dis.drop(dis.index[dis['building'] == 'bridge'], inplace = True)
dis.drop(dis.index[dis['building'] == 'roof'], inplace = True)

#- create a point representing the hole within each building  
#dis['x'] = dis.representative_point().x
#dis['y'] = dis.representative_point().y

#- compute representative point for each geometry
dis["rep_point"] = dis["geometry"].apply(lambda g: g.representative_point() if g else None)
#- extract coordinates
dis["x"] = dis["rep_point"].apply(lambda p: p.x if p else None)
dis["y"] = dis["rep_point"].apply(lambda p: p.y if p else None)
#- drop helper column if you want
dis.drop(columns="rep_point", inplace=True)

hs = dis[['x', 'y', 'ground_height']].copy()


end = time.time()
print('runtime:', str(timedelta(seconds=(end - start))))

runtime: 0:00:00.022653


In [25]:
#print(len(dis), 'buildings have been harvested from the osm.pbf for the', jparams["FocusArea"], 'area') 

In [26]:
dis.head(2)
#dis.plot()

,osm_id,address,building,building:levels,building:use,building:flats,building:units,beds,rooms,residential,...,building_height,roof_height,ground_height,bottom_bridge_height,bottom_roof_height,plus_code,footprint,geometry,x,y
0,6383946,Western Cape Metrorail - Infrastructure Buildi...,office,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,4.1,8.41,4.306250,NaN,NaN,4FRW3FCF+946,"(((266353.2773689979, 6242850.212725159), (266...","POLYGON ((266353.2773689979 6242850.212725159,...",266394.180182,6.242829e+06
1,13328172,None,school,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,6.9,23.80,16.895311,NaN,NaN,4FRW3F96+6MG,"(((265366.0837814461, 6242509.526659243), (265...","POLYGON ((265366.0837814461 6242509.526659243,...",265378.733465,6.242489e+06


## 1. Timing

In [27]:
#- 
dis_c = dis.copy()

In [28]:
#- prepare xyz (more buildings = more time)
start = time.time()

# Convert bytes to DataFrame
xyz_str = xyz_bytes.decode("utf-8")  # Decode to string

dtype_spec = {
    "x": np.float32,  # Reduce precision from float64 to float32 (saves memory)
    "y": np.float32,
    "z": np.float32
}

#df = pd.read_csv(jparams['xyz'], delimiter = ' ', header=None, names=["x", "y", "z"])
#df = pd.read_csv(jparams['xyz'], delimiter=" ", header=None, names=["x", "y", "z"], dtype=dtype_spec, engine="pyarrow")
                                                                                 #, reduced memory, parallelized reading)
df = pd.read_csv(pd.io.common.StringIO(xyz_str), delimiter=" ",  header=None, names=["x", "y", "z"], dtype=dtype_spec)


#- Create the shapely 'geometry' column directly (Vectorized) and GeoDataFrameLite | home-baked gdf
df['geometry'] = df.apply(lambda row: Point(row['x'], row['y']), axis=1) 
gdf = city3D.GeoDataFrameLite(df) 
gdf.crs = epsg
#print(len(gdf))

# --- spatial filtering with STRtree (Replacement for GeoPandas overlay) ---
aoi_geom = shapely.unary_union(aoibuffer['geometry'])
dis_geom = shapely.unary_union(dis_c['geometry'])

mask_geom = aoi_geom.symmetric_difference(dis_geom)
# Note: If mask_geom is a Multi-part geometry, we still need unary_union for STRtree query.
# Here we use the .unary_union property on the result to simplify:
#mask_geom = shapely.unary_union(mask_geom['geometry']) 

#- create the Spatial Index
# This replaces: tree = STRtree(gdf.geometry)
tree = STRtree(gdf['geometry'].values) 

#- query the Tree for Candidates
possible_matches_indices = tree.query(mask_geom, predicate='intersects')#[0]

#- filter the GeoDataFrameLite to candidate points
df_candidates = gdf.iloc[possible_matches_indices]

#- apply the final, exact .within() filter
gdf = df_candidates[df_candidates['geometry'].apply(lambda geom: geom.within(mask_geom))]

# --- cleanup ---
gdf = gdf[gdf['z'] != jparams['nodata']] 
gdf.reset_index(drop=True, inplace=True)
gdf = gdf.round(2)
#print(len(gdf))

end = time.time()
print('runtime:', str(timedelta(seconds=(end - start))))

runtime: 0:04:55.507845


<div class="alert alert-block alert-info"><b>Why is this process taking so long?</b> 
</div>

*-- **What are we doing and why?***
|  | |
|:--------:|:--------:|
|**LoD1**: we only need the surface of the  <br> Earth where there are no buildings.<br><br> **Symmetric Difference**: we are removing all the points, harvested from the DEM, that fall inside a building. <br><br> These are essentially **spatial queries** <br> *—specifically spatial indexing and filtering*. <br><br>  Here we execute `SRTree` for efficient spatial operations. <br><br>  The greater the number of points *(higher resolution elevation model)* and the higher the number of buildings the longer the process will take! |![symDiff.png](_static/symDiff.png)|

In [29]:
#dis.tail(2)

<div class="alert alert-block alert-warning"><b>Prepare for Triangle:</b> </div>

The Python code to execute the `city3D.functions` are in the `city3D.py` script

In [30]:
#- harvest the building vertices. typically the corners. 
ac, c, min_zbld = city3D.getBldVertices(dis, gt_forward, rb)
idx = []
#- segments 
idx, idx01 = city3D.createSgmts(ac, c, gdf, idx)
#- populate the .df with coordinate values (the vertices)
df2 = city3D.concatCoords(gdf, ac)

#- do the same for the area of interest
acoi, ca = city3D.getAOIVertices(aoibuffer, gt_forward, rb)
idx, idx01 = city3D.createSgmts(acoi, ca, df2, idx)
df3 = city3D.concatCoords(df2, acoi)

**Triangle**

In [31]:
pv_pts = df3[['x', 'y', 'z']].values

In [32]:
holes01 = hs[['x', 'y']].round(3).values.tolist()
pts = df3[['x', 'y']].values #, 'z']].values

#the terrain without the blds
A = dict(vertices=pts, segments=idx, holes=holes01)

Tr = tr.triangulate(A, 'p')                  
terrTin = Tr.get('triangles').tolist()

**CityJSON**

In [33]:
#- 
minz = df3['z'].min()
maxz = df3['z'].max()

<div class="alert alert-block alert-warning"><b>create CityJSON</b> </div>

The Python code to execute the `.output_cityjson` function is in the `city3D.py` script

In [34]:
# -- execute function. create CityJSON
crs = epsg[5:]

#city3D.output_cityjson(extent, minz, maxz, terrTin, pv_pts, jparams, min_zbld, acoi, result, crs) 
city3D.output_cityjson(extent, minz, maxz, terrTin, pv_pts, jparams, min_zbld, acoi, crs) 

In [35]:
src_ds = None

<div class="alert alert-block alert-info"><b></b> 

**Go over to [Ninja the online CityJSON viewer](https://ninja.cityjson.org/#) and explore!**

</div>

In [36]:
Tend = time.time()
print('runtime:', str(timedelta(seconds=(Tend - Tstart))))

runtime: 0:05:12.304619


## 2. Quality

<div class="alert alert-block alert-danger"><b>WARNING:</b>  
    
***Lower resolution Elevation Models can leave gaps!***</div>

|  | |
|:--------:|:--------:|
|**25m**|![25.png](_static/25.png)|

<div class="alert alert-block alert-success"><b></b>

**Higher resolution Elevation Models do solve the challenge.** 
</div>

|  | |
|:--------:|:--------:|
|**15m**|![15.png](_static/15.png)|
|**10m**|![10.png](_static/10.png)|

<div class="alert alert-block alert-info"><b>Why is this happening?</b> 
</div>

|  | |
|:--------:|:--------:|
|**What do we** *(in the geospatial community)* **mean  when we say; <br><br> "3D"?**|![3dgis.png](_static/_3DGIS.png)|

|  | |
|:--------:|:--------:|
|We model terrain (a raster DEM) as a 2D surface imbedded in 3D space. <br><br> Each 'xy' coordinate (pixel) only has one 'z' height. This is typically called **2.5D modelling**. <br><br> Notice that *'truthfully'* representing objects connected to a ground surface is impossible. A wall for example could never be straight but have to deviate from the vertical. <br><br> This is why a DEM is often defined as the surface of the earth free of man-made and natural features |![25D.png](_static/_25D.png)|
|To represent a surface *'truthfully'* we can employ **2.75D modelling**. <br><br> The challenge with this solution is; it models the exterior only and it is one surface where objects are one feature. <br><br> A 3D mesh is a 2.75D surface and while traditionally a CAD tool its foray into GIS is recent|![275D.png](_static/_275D.png)|
|Full volumetric **3D modelling**, like a 3D City Model, is actually a 2.5D surface including volumetric 3D objects. <br><br>We can estimate BVPC from these models because we can calculate the volume of a structure|![3D.png](_static/_3D.png)|

<div class="alert alert-block alert-info"><b>Why is this important?</b> 
</div>

We model terrain seperately from the objects (trees, buildings, etc) connected to it.

We remove the buildings, roads, trees, etc. *---we cut them out as we did above--* and due to how *geo3D* creates a city model (terrain modelled seperate from the buildings); the resolution of the raster DEM can create challanges. 
<div class="alert alert-block alert-success"><b></b>

**The challenge is overcome with a finer resolution elevation model.** 
</div>
 

In this particular case a 15m DEM easily solves the challange.